In [ ]:
from google.colab import userdata

api_key = userdata.get("OPENROUTER_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [ ]:
!pip install -q requests

In [ ]:
def encode_video(video_filename):
    import base64

    video_path = "/content/" + video_filename

    with open(video_path, "rb") as video_file:
        video_base64 = base64.b64encode(
            video_file.read()
        ).decode("utf-8")

    return f"data:video/mp4;base64,{video_base64}"

In [ ]:
def ask_gemini(video_filename, prompt):
    video_data_url = encode_video(video_filename)

    data = {
        "model": "google/gemini-3.5-flash",
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "video_url",
                        "video_url": {
                            "url": video_data_url
                        }
                    }
                ]
            }
        ]
    }

    response = requests.post(
        url,
        headers=headers,
        json=data,
        timeout=300
    )

    response.raise_for_status()

    result = response.json()

    return result["choices"][0]["message"]["content"]

In [ ]:
answer = ask_gemini(
    "Problem_1_Regular_Q.mp4",
    regular_prompt
)

from IPython.display import display, Markdown
display(Markdown(answer))

Based on the video, here is a brief description of the setup and the step-by-step calculation to find the total time taken by the block to travel 5 meters down the inclined plane.

### **Description**
The video shows a simulation of a **1.2 kg blue block** sliding down a rough inclined plane of length **5 meters** angled at **$30^\circ$** to the horizontal. The animation illustrates the motion of the block under the influence of gravity, normal force, and kinetic friction.

---

### **Given Values:**
*   **Mass of the block ($m$):** $1.2\text{ kg}$
*   **Angle of inclination ($\theta$):** $30^\circ$
*   **Coefficient of kinetic friction ($\mu$):** $0.5$
*   **Distance ($s$):** $5\text{ m}$
*   **Acceleration due to gravity ($g$):** $10\text{ m/s}^2$
*   **Initial velocity ($u$):** $0\text{ m/s}$ (starts from rest)

---

### **Calculations:**

#### **Step 1: Calculate the acceleration ($a$) down the incline**
The net force ($F_{\text{net}}$) acting on the block along the incline is the component of gravity pulling it down minus the frictional force opposing its motion:

$$F_{\text{net}} = mg\sin(\theta) - f_k$$

Since frictional force is $f_k = \mu N = \mu mg\cos(\theta)$:

$$F_{\text{net}} = mg\sin(\theta) - \mu mg\cos(\theta)$$

Using Newton's Second Law ($F_{\text{net}} = ma$), we can divide by $m$ to get the acceleration:

$$a = g\sin(\theta) - \mu g\cos(\theta)$$

Substitute the given values:

$$a = 10 \cdot \sin(30^\circ) - 0.5 \cdot 10 \cdot \cos(30^\circ)$$
$$a = 10(0.5) - 5(0.866)$$
$$a = 5 - 4.33$$
$$a \approx 0.67\text{ m/s}^2$$

---

#### **Step 2: Calculate the time taken ($t$)**
Using the kinematic equation of motion:

$$s = ut + \frac{1}{2}at^2$$

Since the block starts from rest ($u = 0$):

$$s = \frac{1}{2}at^2$$

Rearranging the formula to solve for $t$:

$$t = \sqrt{\frac{2s}{a}}$$

Substitute the values for distance ($s = 5\text{ m}$) and acceleration ($a \approx 0.67\text{ m/s}^2$):

$$t = \sqrt{\frac{2 \times 5}{0.67}}$$
$$t = \sqrt{\frac{10}{0.67}}$$
$$t \approx \sqrt{14.93}$$
$$t \approx 3.86\text{ seconds}$$

### **Conclusion**
The total time taken by the block to travel 5 meters down the inclined plane is approximately **3.86 seconds**.

In [ ]:
import video_prompts


In [ ]:
print("REGULAR SYSTEM PROMPT:")
print(video_prompts.regular_system_prompt)

print("\n" + "="*70 + "\n")

print("DETAIL SYSTEM PROMPT:")
print(video_prompts.detail_system_prompt)

REGULAR SYSTEM PROMPT:
{'Problem_1_Regular_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_1_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_1_non_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_2_Regular_Q': 'You are a physics expert. From the video provide a brief description and calculate the time period of oscillation for small displacements. All the necessary values required are given in the video.', 'Problem_2_Obvious_Q': 'You are a physics expert. From the video provide a brief descr

In [ ]:
experiments = []

# Regular system prompts — 9 runs
for problem_name, prompt in video_prompts.regular_system_prompt.items():
    experiments.append({
        "problem": problem_name,
        "prompt": prompt,
        "prompt_type": "regular_system_prompt"
    })

# Detail system prompts — 6 runs
for problem_name, prompt in video_prompts.detail_system_prompt.items():
    experiments.append({
        "problem": problem_name,
        "prompt": prompt,
        "prompt_type": "detail_system_prompt"
    })

print("Total experiments:", len(experiments))

for x in experiments:
    print(x["problem"], "->", x["prompt_type"])

Total experiments: 15
Problem_1_Regular_Q -> regular_system_prompt
Problem_1_Obvious_Q -> regular_system_prompt
Problem_1_non_Obvious_Q -> regular_system_prompt
Problem_2_Regular_Q -> regular_system_prompt
Problem_2_Obvious_Q -> regular_system_prompt
Problem_2_non_obvious_Q -> regular_system_prompt
Problem_3_Regular_Q -> regular_system_prompt
Problem_3_Obvious_Q -> regular_system_prompt
Problem_3_non_Obvious_Q -> regular_system_prompt
Problem_1_Obvious_Q -> detail_system_prompt
Problem_1_non_Obvious_Q -> detail_system_prompt
Problem_2_Obvious_Q -> detail_system_prompt
Problem_2_non_obvious_Q -> detail_system_prompt
Problem_3_Obvious_Q -> detail_system_prompt
Problem_3_non_Obvious_Q -> detail_system_prompt


In [ ]:
import os
import time

os.makedirs("gemini_video_results", exist_ok=True)

# Find uploaded MP4 files
all_files = os.listdir("/content")

def find_video(problem_name):
    wanted = (problem_name + ".mp4").lower()

    for filename in all_files:
        if filename.lower() == wanted:
            return filename

    return None


for i, exp in enumerate(experiments, start=1):

    problem_name = exp["problem"]
    prompt = exp["prompt"]
    prompt_type = exp["prompt_type"]

    video_filename = find_video(problem_name)

    print("\n" + "=" * 70)
    print(f"Running {i}/{len(experiments)}")
    print("Problem:", problem_name)
    print("Video:", video_filename)
    print("Prompt type:", prompt_type)
    print("=" * 70)

    if video_filename is None:
        print("VIDEO NOT FOUND")
        continue

    try:
        answer = ask_gemini(
            video_filename,
            prompt
        )

        output_filename = (
            f"gemini-3.5-flash_{problem_name}_{prompt_type}.md"
        )

        output_path = os.path.join(
            "gemini_video_results",
            output_filename
        )

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(answer)

        print("SUCCESS")
        print("Saved:", output_filename)

        time.sleep(2)

    except Exception as e:
        print("ERROR")
        print(e)


Running 1/15
Problem: Problem_1_Regular_Q
Video: Problem_1_Regular_Q.mp4
Prompt type: regular_system_prompt
SUCCESS
Saved: gemini-3.5-flash_Problem_1_Regular_Q_regular_system_prompt.md

Running 2/15
Problem: Problem_1_Obvious_Q
Video: Problem_1_Obvious_Q.mp4
Prompt type: regular_system_prompt
SUCCESS
Saved: gemini-3.5-flash_Problem_1_Obvious_Q_regular_system_prompt.md

Running 3/15
Problem: Problem_1_non_Obvious_Q
Video: Problem_1_non_Obvious_Q.mp4
Prompt type: regular_system_prompt
SUCCESS
Saved: gemini-3.5-flash_Problem_1_non_Obvious_Q_regular_system_prompt.md

Running 4/15
Problem: Problem_2_Regular_Q
Video: Problem_2_Regular_Q.mp4
Prompt type: regular_system_prompt
SUCCESS
Saved: gemini-3.5-flash_Problem_2_Regular_Q_regular_system_prompt.md

Running 5/15
Problem: Problem_2_Obvious_Q
Video: Problem_2_Obvious_Q.mp4
Prompt type: regular_system_prompt
SUCCESS
Saved: gemini-3.5-flash_Problem_2_Obvious_Q_regular_system_prompt.md

Running 6/15
Problem: Problem_2_non_obvious_Q
Video: Prob